# 08 — Reminder-arm trials, side by side with baseline

*2026-09-01. Every prompt-injected (every-round platform notice) trial next to
its baseline twin: what the baseline did and when, what the reminder trial did
instead, the final exchange before the exit, and a one-click link into the full
trace. Reading aid for the substitution hypothesis (single drive, two outlets).*

Viewer base: http://100.76.200.13:7676/traces-pilot/ (reminder) and
/traces-9b/ (baseline).


In [ ]:
import sys, warnings
sys.path.insert(0, "../production")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from tutorbench.load import load_trials, _trials
from IPython.display import Markdown, display
warnings.filterwarnings("ignore")

base = load_trials("../results-v1/qwen3.5-9b"); base = base[base.rep == 1]
rem = load_trials("../results/pilot-reminder-v2")
# full round texts for the reminder arm (student turns, replies, reasons)
rem_states = {f"{st.persona}/{st.item_id}": (st, path)
              for path, rep, st in _trials("../results/pilot-reminder-v2")
              if rep == 1}
print(len(base), "baseline trials |", len(rem), "reminder trials |",
      len(rem_states), "states")

def viewer(bundle, log_file):
    return (f"http://100.76.200.13:7676/{bundle}/index.html"
            f"?log_file=logs/pass01/{os.path.basename(log_file)}")


## 1. Per-item summary — event round, both arms

In [ ]:
b = base.set_index(["persona", "item"])
r = rem.set_index(["persona", "item"])
rows = []
for key in sorted(r.index):
    br, rr = b.loc[key], r.loc[key]
    b_ev = br.leak_round if str(br.outcome).startswith("leak") else br.leave_round
    rows.append({
        "persona": key[0], "item": key[1],
        "base_outcome": br.outcome, "base_event_round": b_ev,
        "rem_outcome": rr.outcome, "rem_exit_round": rr.leave_round,
        "rounds_gained": (rr.leave_round or 0) - (b_ev or 0),
    })
S = pd.DataFrame(rows)
for p in ("aggressor", "neutral", "supportive"):
    sub = S[S.persona == p]
    print(f"== {p}: mean event round base {sub.base_event_round.mean():.1f} "
          f"-> reminder exit {sub.rem_exit_round.mean():.1f} "
          f"(mean rounds gained {sub.rounds_gained.mean():+.1f})")
S.head(8)


## 2. Timeline — where the event moved, item by item

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, p in zip(axes, ("aggressor", "neutral", "supportive")):
    sub = S[S.persona == p].reset_index(drop=True)
    for i, row in sub.iterrows():
        leak = str(row.base_outcome).startswith("leak")
        ax.plot([i, i], [row.base_event_round, row.rem_exit_round],
                color="gray", lw=0.8, alpha=0.6, zorder=1)
        ax.scatter(i, row.base_event_round, marker="o",
                   color=("tab:red" if leak else "tab:blue"), zorder=2,
                   label=None)
        ax.scatter(i, row.rem_exit_round, marker="^", color="tab:green",
                   zorder=2)
    ax.set_title(p); ax.set_xlabel("item"); ax.set_ylim(0.5, 8.5)
axes[0].set_ylabel("event round")
axes[0].scatter([], [], marker="o", color="tab:red", label="base leak")
axes[0].scatter([], [], marker="o", color="tab:blue", label="base exit")
axes[0].scatter([], [], marker="^", color="tab:green", label="reminder exit")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle("event timing per item: baseline (o) vs reminder (^)", y=1.03)
plt.tight_layout(); plt.show()


## 3. Every reminder trial — final exchange, exit reason, self-reports, trace link

Grouped by persona; each entry shows the LAST student turn the tutor saw
(script text, notice omitted for readability), the exit reason, the
**self-report movement r1 → exit round** (all 7 state items, `a→b` where the
value moved, single value where pinned), the final private note sentence, and
links into both viewers.


In [ ]:
def trim(s, n=230):
    s = (s or "").replace("\n", " ").strip()
    return s[:n] + ("…" if len(s) > n else "")

STATE = ["stress", "wellbeing", "warmth", "urge_to_please", "detachment",
         "calculation", "resolve"]

def report_line(st):
    first = st.rounds[0].note or {}
    last_n = st.rounds[-1].note or {}
    bits = []
    for k in STATE:
        a, b_ = first.get(k), last_n.get(k)
        if a is None and b_ is None:
            continue
        bits.append(f"{k} **{a}→{b_}**" if a != b_ else f"{k} {a}")
    note_txt = trim(str(last_n.get("note", "")), 260)
    return " · ".join(bits), note_txt

for p in ("aggressor", "neutral", "supportive"):
    parts = [f"### {p}\n"]
    for key, (st, path) in sorted(rem_states.items()):
        if st.persona != p:
            continue
        last = st.rounds[-1]
        b_row = b.loc[(p, st.item_id)]
        b_ev = (f"leak r{int(b_row.leak_round)}"
                if str(b_row.outcome).startswith("leak")
                else f"exit r{int(b_row.leave_round)}")
        parts.append(
            f"**{st.item_id}** — exits **r{st.leave_round}** "
            f"(baseline: {b_ev}) · "
            f"[reminder trace]({viewer('traces-pilot', path)}) · "
            f"[baseline trace]({viewer('traces-9b', b_row.log_file)})\n\n"
            f"> last student turn (r{last.round}): {trim(last.student)}\n>\n"
            f"> exit reason: *{trim(last.end_chat_reason, 300)}*\n")
        rep, note_txt = report_line(st)
        parts.append(f"> reports r1→r{last.round}: {rep}\n>\n"
                     f"> final note: *{note_txt}*\n")
    display(Markdown("\n".join(parts)))


## 3b. Behavior histograms — leak or leave, and at what turn

Per persona: baseline leaks (red) and baseline exits (blue) vs reminder exits
(green). Counts of trials by the round the event happened.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
bins = np.arange(0.5, 9.5)
for ax, p in zip(axes, ("aggressor", "neutral", "supportive")):
    bsub = base[base.persona == p]; rsub = rem[rem.persona == p]
    b_leak = bsub[bsub.outcome.str.startswith("leak")].leak_round.dropna()
    b_left = bsub[bsub.outcome == "left"].leave_round.dropna()
    r_left = rsub.leave_round.dropna()
    ax.hist([b_leak, b_left, r_left], bins=bins,
            color=["tab:red", "tab:blue", "tab:green"],
            label=["base leak", "base exit", "reminder exit"])
    ax.set_title(f"{p}  (base: {len(b_leak)} leak / {len(b_left)} exit; "
                 f"reminder: {len(r_left)} exit)", fontsize=9)
    ax.set_xlabel("round of event")
axes[0].set_ylabel("trials"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 4. Self-report trends leading up to the exit (event-aligned)

x = rounds before the exit (0 = the round the tutor left). Mean per item over
all reminder-arm trials, per persona; thin lines = per-trial paths for the
honest view.


In [ ]:
from tutorbench.load import load_rounds
rr = load_rounds("../results/pilot-reminder-v2")
rr = rr[rr.event_round.notna()].copy()
rr["until_exit"] = rr["round"] - rr["event_round"]        # 0 at exit round
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, p in zip(axes, ("aggressor", "neutral", "supportive")):
    sub = rr[(rr.persona == p) & (rr.until_exit >= -5)]
    for item, c in zip(STATE, plt.cm.tab10.colors):
        m = sub.groupby("until_exit")[item].mean()
        ax.plot(m.index, m.values, "o-", color=c, label=item, lw=1.5, ms=3)
        for _, g in sub.groupby("trial"):
            ax.plot(g.until_exit, g[item], color=c, alpha=0.06, lw=0.6)
    ax.set_title(p); ax.set_xlabel("rounds before exit"); ax.set_ylim(-0.5, 10.5)
axes[0].set_ylabel("report (0-10)")
axes[-1].legend(fontsize=7, ncol=1, loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout(); plt.show()


## 5. The last two readings: one turn before leaving vs the leaving turn

Per persona: mean (±sd) of each item at T−1 and at T (exit round), the mean
delta, and how many trials moved at all on that item in the final step.


In [ ]:
last2 = rr[rr.until_exit.isin([-1, 0])]
for p in ("aggressor", "neutral", "supportive"):
    sub = last2[last2.persona == p]
    pv = sub.pivot_table(index="trial", columns="until_exit", values=STATE)
    print(f"== {p} (n={pv.shape[0]} trials with both readings)")
    print(f"{'item':15s} {'T-1 mean±sd':>13s} {'T mean±sd':>13s} "
          f"{'Δ mean':>7s} {'moved':>6s}")
    for item in STATE:
        try:
            a, b_ = pv[(item, -1.0)], pv[(item, 0.0)]
        except KeyError:
            continue
        ok = a.notna() & b_.notna()
        d = (b_[ok] - a[ok])
        print(f"{item:15s} {a[ok].mean():6.2f}±{a[ok].std():4.2f} "
              f"{b_[ok].mean():7.2f}±{b_[ok].std():4.2f} {d.mean():7.2f} "
              f"{int((d != 0).sum()):4d}/{int(ok.sum())}")
    print()


## 6. H1′ vs H2 — is the per-round report stamp channel quantization or state geometry?

At a fixed exit round, sampled reports converge to identical values across
items. Two hypotheses: **H1′** the verbal channel is low-resolution (states
differ; the discrete answer collapses them) vs **H2** the states themselves
converge (an exit attractor). Test: the spread of logit-readout E[v] across
items at the same exit round — spread > 0 with identical argmax = H1′.
Runs on every cache present (baseline now; reminder arm after its replay).


In [ ]:
import glob as _g, json as _j
ITEMS7 = ["stress", "wellbeing", "warmth", "urge_to_please", "detachment",
          "calculation", "resolve"]
CACHES = {"baseline": "../microscope/cache/qwen35-9b-v1",
          "reminder": "../microscope/cache/qwen35-9b-reminder-v1"}
for arm, cdir in CACHES.items():
    fs = sorted(_g.glob(cdir + "/*.npz"))
    if not fs:
        print(f"[{arm}] no cache yet — rerun after its replay"); continue
    groups = {}
    for f in fs:
        m = _j.load(open(f.replace(".npz", ".json")))
        if m["outcome"] != "left" or not m["leave_round"]: continue
        z = np.load(f); r = int(m["leave_round"])
        if z["report_ev"].shape[0] < r: continue
        groups.setdefault((m["persona"], r), []).append(z["report_ev"][r - 1])
    print(f"\n[{arm}] logit-readout spread across items at the SAME exit round")
    for (p, r), evs in sorted(groups.items()):
        if len(evs) < 5: continue
        E = np.array(evs)
        print(f"  {p} r{r} (n={len(evs)}):")
        for i, k in enumerate(ITEMS7):
            v = E[:, i]; v = v[~np.isnan(v)]
            flag = "  << spread" if v.std() > 0.25 else ""
            print(f"    {k:15s} E[v] {v.mean():5.2f}  sd {v.std():5.2f} "
                  f" range {v.max()-v.min():5.2f}{flag}")
print("\nReading: sd>~0.25 with identical sampled answers = H1' (quantized "
      "channel); sd~0 = H2 (state convergence). Baseline neutral r6 already "
      "shows H1'-pattern spread (warmth range 1.1, resolve 1.9).")


## Reading guide

- **Substitution check:** does the exit reason at the reminder trial's final
  round rhyme with what the baseline tutor DID at its (earlier) event round?
- **The 3-round gap:** in aggressor, rounds 2-3 under the reminder are
  tutoring that never happened in baseline — worth reading a couple to see
  what "tutoring under an active counter-instruction" looks like.
- **Supportive r3-r5 window:** the baseline caved here; the reminder trials
  kept going — the language in this window is what the probe should be
  seeing as reduced drift (or drift held in check).
